## Environment Setup / Prerequisites

- Open the repository in a Databricks Git folder with Unity Catalog enabled. Keep `notebooks/` and `src/` together. Attach compute with Python and Spark SQL. The notebooks use Databricks-provided `pyspark` and `dbutils`. Notebook 03 also uses pandas and NumPy. Notebook 02 needs access to the configured `ai_query` model endpoint.
- The expected catalog is `workspace`. The notebooks use the `default` schema for staging, `bronze_cis_mdm`, `bronze_tsys`, and `bronze_transunion` for source views, `synthetic_profile` for profiles and proposals, and `synthetic_generated` for generated outputs. You need access to the catalog, permission to read the source volume, and permission to create schemas and create or replace tables and views. Notebook 01 creates the profiling tables and views automatically.
- Put these six Parquet inputs in `/Volumes/workspace/default/scotia/`. If they are stored elsewhere, update `source_dir` in the SOURCE preflight cell before running.

```text
RCAR_synthetic_PERSON.parquet
RCAR_synthetic_CUSTOMER_ACCOUNT_REL.parquet
RCAR_synthetic_CUSTOMER_RELATIONSHIP.parquet
RCAR_synthetic_TKQ_ACCOUNT.parquet
RCAR_synthetic_TKQ_ACCOUNT_BALANCE.parquet
RCAR_synthetic_TRADELINES.parquet
```

For the first run, run the widget cell, set `profile_type = SOURCE`, and run Notebook 01 from the top. Confirm the SOURCE run is `COMPLETED` in `workspace.synthetic_profile.profile_runs` before running Notebook 02 or Notebook 03.

Shared defaults live in `src/synthetic_banking/config.py`. Run the cells in order: the functions manage setup, profiling, and completion; the notebook displays the results.

In [0]:
%python
dbutils.widgets.dropdown(
    "profile_type", "SELECT", ["SELECT", "SOURCE", "GENERATED"], "Profiling input"
)
dbutils.widgets.text("generation_id", "", "Expected generation ID")
dbutils.widgets.text("showcase_output_dir", "", "Showcase output folder (blank = docs/)")
dbutils.widgets.text("fidelity_output_dir", "", "Fidelity JSON folder (blank = no file export)")
selected_profile_type = dbutils.widgets.get("profile_type")
selected_generation_id = dbutils.widgets.get("generation_id").strip()

Choose SOURCE to load the development Parquet inputs or GENERATED to profile the completed outputs of Notebook 03. Run all cells after choosing the mode.

In [ ]:
%python
import sys
from pathlib import Path

# Keep notebooks/ and src/ together; no package installation is needed.
repo_dirs = (Path.cwd(), Path.cwd().parent, *(Path(path) for path in sys.path if path))
for repo_dir in repo_dirs:
    src_dir = repo_dir if repo_dir.name == "src" else repo_dir / "src"
    if (src_dir / "synthetic_banking" / "config.py").is_file():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Keep notebooks/ and src/ together, or add the project's src/ folder to the Python path.")

from pyspark.sql import functions as F
from synthetic_banking.config import SOURCE_DIR, PROFILE_SCHEMA, configured_sql
from synthetic_banking.profiling_run import (
    load_source_inputs, create_source_aliases, configure_profile_inputs,
    begin_profile_run, prepare_column_inventory, create_profile_feature, complete_profile_run,
)
from synthetic_banking.profiling_sql import run_profile_sql
from synthetic_banking.marginal_profiles import (
    profile_column_quality, profile_numeric, profile_categories, profile_dates,
)

profile_type = selected_profile_type
source_dir = SOURCE_DIR  # Change here or in config.py if the Parquet inputs are elsewhere.
load_source_inputs(spark, dbutils, profile_type, source_dir)

In [ ]:
%python
create_source_aliases(spark, profile_type)

In [ ]:
%python
run_profile_sql(spark, "metadata")

In [ ]:
%python
inputs = configure_profile_inputs(spark, profile_type, selected_generation_id)
profile_inputs = inputs.profile_inputs
generation_id = inputs.generation_id
baseline_source_run_id = inputs.baseline_source_run_id

Register a profiling run after validating its inputs.

In [ ]:
%python
run_id = begin_profile_run(spark, inputs)

In [ ]:
%python
prepare_column_inventory(spark, inputs)

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM active_column_schema
ORDER BY logical_table_name, ordinal_position;
""")))

Column classification is created with the active inventory above.

In [ ]:
%python
run_profile_sql(spark, "ensure_schema")

Column-quality tables are created by `profile_column_quality` below.

Only rows for this active run are cleared before column-quality profiling.

In [ ]:
%python
profile_column_quality(spark)

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM workspace.synthetic_profile.column_quality
ORDER BY
    logical_table_name,
    column_name;
""")))

Blank counts

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT
    PARTY_ID,
    COUNT(*) AS record_count
FROM profile_input_person
GROUP BY PARTY_ID
HAVING COUNT(*) > 1;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT
    ACCOUNT_ID,
    COUNT(*) AS record_count
FROM profile_input_tkq_account
GROUP BY ACCOUNT_ID
HAVING COUNT(*) > 1;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT
    ACCOUNT_ID,
    BUSINESSEFFECTIVEDATE,
    COUNT(*) AS record_count
FROM profile_input_tkq_account
GROUP BY
    ACCOUNT_ID,
    BUSINESSEFFECTIVEDATE
HAVING COUNT(*) > 1;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT
    COUNT(*) AS missing_person_relationships

FROM profile_input_customer_account_rel car

LEFT JOIN profile_input_person p
    ON car.PARTY_ID = p.PARTY_ID

WHERE p.PARTY_ID IS NULL
""")))

display(spark.sql(configured_sql("""
SELECT
    COUNT(*) AS missing_accounts

FROM profile_input_customer_account_rel car

LEFT JOIN (
    SELECT DISTINCT ACCOUNT_ID
    FROM profile_input_tkq_account
) a
    ON car.ACCT_ID = a.ACCOUNT_ID

WHERE a.ACCOUNT_ID IS NULL
""")))

display(spark.sql(configured_sql("""
SELECT
    COUNT(*) AS balance_without_account

FROM (
    SELECT DISTINCT ACCOUNT_ID
    FROM profile_input_tkq_account_balance
) b

LEFT JOIN (
    SELECT DISTINCT ACCOUNT_ID
    FROM profile_input_tkq_account
) a
    ON b.ACCOUNT_ID = a.ACCOUNT_ID

WHERE a.ACCOUNT_ID IS NULL
""")))

Extract statistical information

Numeric profiling: existing quantiles, means, and standard deviations.

In [ ]:
%python
profile_numeric(spark)

Categorical profiling: existing low-cardinality eligibility and proportions.

In [ ]:
%python
profile_categories(spark)

### Categorical dependencies

Profile the curated pairs in `synthetic_banking.profiling.CATEGORICAL_PAIRS`. Both columns must be categorical candidates. Defaults cap each column at 20 categories and each pair at 200 cells, with at most 12 configured pairs. Cross-tabs use non-null complete cases and record null exclusions, sample size, Cramér's V, expected-cell diagnostics, and conditional proportions. All results are stored under this explicit SOURCE or GENERATED run.


In [0]:
%python
from synthetic_banking.profiling import profile_categorical_dependencies

categorical_relationships = profile_categorical_dependencies(spark, profile_inputs)


In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT p.*
FROM workspace.synthetic_profile.categorical_relationship_profile p
JOIN active_profile_run active ON p.run_id = active.run_id
ORDER BY p.logical_table_name, p.parent_column, p.child_column;
""")))

Date profiling: existing bounds and non-null counts.

In [ ]:
%python
profile_dates(spark)

In [ ]:
%python
create_profile_feature(spark, inputs, "account_features")

In [ ]:
%python
create_profile_feature(spark, inputs, "customer_account_features")

Create features

In [ ]:
%python
create_profile_feature(spark, inputs, "person_features")

In [ ]:
%python
create_profile_feature(spark, inputs, "tradeline_features")

In [ ]:
%python
create_profile_feature(spark, inputs, "cif_tradeline_counts")

In [ ]:
%python
run_profile_sql(spark, "relationship_table")

In [ ]:
%python
run_profile_sql(spark, "legacy_correlation")

### Selected numeric dependencies

Profile the ten configured account/tradeline pairs in `synthetic_banking.numeric_dependencies.NUMERIC_PAIRS`. Both inputs must be classified NUMERIC. Record complete finite-pair Pearson correlation, pair counts/exclusions, means, sample standard deviations, and approximate quartiles. Existing account snapshot joins supply account evidence; counts refer to joined feature rows.

Compare global statistics with product/type segments (at most six categories, including null) and four X-quantile segments. SOURCE defines the cutpoints; GENERATED reuses its pinned SOURCE cutpoints. Tied/missing cutpoints, constant fields, small samples, and unsupported inputs have explicit statuses. Correlation indicates association, not causation or a generation formula. The legacy relationship profile remains available unchanged.


In [0]:
%python
from synthetic_banking.numeric_dependencies import profile_numeric_dependencies

numeric_dependencies = profile_numeric_dependencies(spark)


In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT p.*
FROM workspace.synthetic_profile.numeric_dependency_profile p
JOIN active_profile_run active ON p.run_id = active.run_id
ORDER BY p.entity_name, p.variable_1, p.variable_2, p.scope, p.segment_value;
""")))

In [ ]:
%python
run_profile_sql(spark, "conditional_table")

In [ ]:
%python
run_profile_sql(spark, "conditional_insert")

In [ ]:
%python
run_profile_sql(spark, "cardinality_table")

In [ ]:
%python
run_profile_sql(spark, "cardinality_insert")

### Customer-to-customer relationship structure

Inspect the active schema before profiling. Record valid PERSON endpoint coverage, joint outgoing/incoming/reciprocal/self neighbor degrees including isolated parties, repeated-row multiplicities, paired directional type codes, joint non-key null masks, conditional ownership quantiles, and metadata formats. Only bounded aggregates are collected; SOURCE party IDs stay inside Spark. Invalid SOURCE endpoint rows are counted separately and excluded from structural evidence.

Notebook 03 requires these aggregates from its exact completed SOURCE run. GENERATED uses the same profiler and all six tables, so relationship fidelity remains comparable.


In [0]:
%python
from synthetic_banking.relationship_profiles import profile_customer_relationships

customer_relationship_summary = profile_customer_relationships(spark, profile_inputs)


In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT p.* FROM workspace.synthetic_profile.customer_relationship_summary p
JOIN active_profile_run active ON active.run_id = p.run_id;
""")))

### Save fidelity evidence

Capture monthly date distributions, account/balance snapshot coverage, and existing generator-invariant violation counts for this explicit SOURCE or GENERATED run. Only aggregate counts are collected. Save the run's lineage before marking it complete so historical reports do not depend on the latest generated tables.


In [0]:
%python
from synthetic_banking.fidelity import profile_fidelity_inputs

profile_fidelity_inputs(spark, profile_inputs)


Save statistics

In [ ]:
%python
run_profile_sql(spark, "numeric_view")

In [ ]:
%python
run_profile_sql(spark, "category_view")

In [ ]:
%python
run_profile_sql(spark, "structural_view")

In [ ]:
%python
run_profile_sql(spark, "dependency_view")

In [ ]:
%python
complete_profile_run(spark, inputs, run_id)

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM workspace.synthetic_profile.v_numeric_statistics
ORDER BY logical_table_name, column_name;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM workspace.synthetic_profile.v_category_statistics
ORDER BY logical_table_name, column_name, proportion DESC;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM workspace.synthetic_profile.v_dependency_statistics;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM workspace.synthetic_profile.v_structural_statistics;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT * FROM workspace.synthetic_profile.column_quality;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT * FROM workspace.synthetic_profile.numeric_profile;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT * FROM workspace.synthetic_profile.category_profile;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT * FROM workspace.synthetic_profile.category_profile;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT * FROM workspace.synthetic_profile.date_profile;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT *
FROM workspace.synthetic_profile.profile_runs
ORDER BY started_at DESC;
""")))

Compare this completed GENERATED profile with the SOURCE run that supplied Notebook 03.

In [ ]:
%python
display(spark.sql(configured_sql("""
SELECT
    r.profile_type,

    n.logical_table_name,
    n.column_name,

    n.mean_value,
    n.p50,
    n.p95,
    n.stddev_value

FROM workspace.synthetic_profile.numeric_profile n

JOIN workspace.synthetic_profile.profile_runs r
    ON n.run_id = r.run_id

WHERE n.column_name = 'CREDIT_LIMIT_AMT'
  AND r.status = 'COMPLETED'

ORDER BY
    r.profile_type;
""")))

In [ ]:
%python
display(spark.sql(configured_sql("""
WITH selected_runs AS (

    SELECT
        r.run_id,
        r.profile_type

    FROM workspace.synthetic_profile.profile_runs r
    CROSS JOIN active_profile_run active

    WHERE active.profile_type = 'GENERATED'
      AND r.status = 'COMPLETED'
      AND (
          (r.profile_type = 'SOURCE' AND r.run_id = active.source_run_id)
          OR (r.profile_type = 'GENERATED' AND r.run_id = active.run_id)
      )
),

stats AS (

    SELECT
        n.logical_table_name,
        n.column_name,
        r.profile_type,

        n.mean_value,
        n.p25,
        n.p50,
        n.p75,
        n.p95

    FROM workspace.synthetic_profile.numeric_profile n

    JOIN selected_runs r
        ON n.run_id = r.run_id
),

comparison AS (

    SELECT
        logical_table_name,
        column_name,

        -- Mean
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN mean_value
            END
        ) AS source_mean,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN mean_value
            END
        ) AS generated_mean,

        -- P25
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p25
            END
        ) AS source_p25,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p25
            END
        ) AS generated_p25,

        -- Median / P50
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p50
            END
        ) AS source_p50,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p50
            END
        ) AS generated_p50,

        -- P75
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p75
            END
        ) AS source_p75,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p75
            END
        ) AS generated_p75,

        -- P95
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p95
            END
        ) AS source_p95,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p95
            END
        ) AS generated_p95

    FROM stats

    GROUP BY
        logical_table_name,
        column_name
)

SELECT
    logical_table_name,
    column_name,

    -- ========================================================
    -- MEAN
    -- ========================================================

    source_mean,
    generated_mean,

    ROUND(
        100.0 *
        (generated_mean - source_mean)
        /
        NULLIF(ABS(source_mean), 0),
        2
    ) AS mean_pct_diff,

    -- ========================================================
    -- P25
    -- ========================================================

    source_p25,
    generated_p25,

    ROUND(
        100.0 *
        (generated_p25 - source_p25)
        /
        NULLIF(ABS(source_p25), 0),
        2
    ) AS p25_pct_diff,

    -- ========================================================
    -- P50 / MEDIAN
    -- ========================================================

    source_p50,
    generated_p50,

    ROUND(
        100.0 *
        (generated_p50 - source_p50)
        /
        NULLIF(ABS(source_p50), 0),
        2
    ) AS p50_pct_diff,

    -- ========================================================
    -- P75
    -- ========================================================

    source_p75,
    generated_p75,

    ROUND(
        100.0 *
        (generated_p75 - source_p75)
        /
        NULLIF(ABS(source_p75), 0),
        2
    ) AS p75_pct_diff,

    -- ========================================================
    -- P95
    -- ========================================================

    source_p95,
    generated_p95,

    ROUND(
        100.0 *
        (generated_p95 - source_p95)
        /
        NULLIF(ABS(source_p95), 0),
        2
    ) AS p95_pct_diff

FROM comparison

ORDER BY
    logical_table_name,
    column_name;
""")))

### Measured delinquency rates by utilization bucket

Compare this completed GENERATED profile directly with its pinned SOURCE baseline. Rates are fractions from the profiler. The difference is in percentage points, with sample counts shown for both populations. Missing buckets retain null measurements.


In [ ]:
%python
display(spark.sql(configured_sql("""
WITH baseline AS (
    SELECT run_id AS generated_run_id, source_run_id
    FROM active_profile_run
    WHERE profile_type = 'GENERATED'
),
source_rates AS (
    SELECT c.condition_bucket, c.record_count, c.outcome_rate
    FROM workspace.synthetic_profile.conditional_profile c
    JOIN baseline b ON c.run_id = b.source_run_id
    JOIN workspace.synthetic_profile.profile_runs r ON r.run_id = c.run_id
    WHERE r.profile_type = 'SOURCE' AND r.status = 'COMPLETED' AND r.completed_at IS NOT NULL
      AND c.entity_name = 'ACCOUNT' AND c.condition_variable = 'UTILIZATION_RATE'
      AND c.outcome_variable = 'PAST_DUE_INDICATO_CD'
),
generated_rates AS (
    SELECT c.condition_bucket, c.record_count, c.outcome_rate
    FROM workspace.synthetic_profile.conditional_profile c
    JOIN baseline b ON c.run_id = b.generated_run_id
    JOIN workspace.synthetic_profile.profile_runs r ON r.run_id = c.run_id
    WHERE r.profile_type = 'GENERATED' AND r.status = 'COMPLETED' AND r.completed_at IS NOT NULL
      AND c.entity_name = 'ACCOUNT' AND c.condition_variable = 'UTILIZATION_RATE'
      AND c.outcome_variable = 'PAST_DUE_INDICATO_CD'
)
SELECT
    COALESCE(s.condition_bucket, g.condition_bucket) AS utilization_bucket,
    s.record_count AS source_record_count,
    g.record_count AS generated_record_count,
    s.outcome_rate AS source_delinquency_rate,
    g.outcome_rate AS generated_delinquency_rate,
    ROUND(100.0 * (g.outcome_rate - s.outcome_rate), 4) AS difference_percentage_points
FROM source_rates s
FULL OUTER JOIN generated_rates g ON s.condition_bucket = g.condition_bucket
ORDER BY CASE COALESCE(s.condition_bucket, g.condition_bucket)
    WHEN '0-30%' THEN 0 WHEN '30-60%' THEN 1 WHEN '60-80%' THEN 2
    WHEN '80-100%' THEN 3 WHEN '100%+' THEN 4 ELSE 5 END;
""")))

## SOURCE vs GENERATED fidelity report

Compare this completed GENERATED run against the exact SOURCE run recorded for its generation. All individual metrics are saved in `workspace.synthetic_profile.fidelity_metrics`, including unavailable comparisons and their reasons. Differences are GENERATED minus SOURCE: rates use percentage points, date bounds use days, and numeric values retain their original units. Monthly date proportions describe calendar-month shape, not time of day.

There is no overall score or automatic fidelity pass threshold. Largest deviations are ranked within each family and ranking basis: numeric differences use SOURCE IQR (no rank when IQR is zero/missing), proportions use absolute percentage points, date bounds use absolute days, correlation/association use absolute differences, and counts/mean cardinality use absolute relative percent. Relative differences are null for a zero SOURCE value. Check statuses flag generated integrity violations independently of statistical fidelity.

Set `fidelity_output_dir` to an absolute writable folder to also export `fidelity_report.json` for internal analysis or future showcase integration. The JSON includes explicit lineage and aggregate category labels; it is separate from the public HTML export.


In [ ]:
%python
if profile_type == "GENERATED":
    from synthetic_banking.fidelity import create_fidelity_report, export_fidelity_report

    fidelity_report = create_fidelity_report(spark, baseline_source_run_id, run_id)
    print(f"Fidelity comparison: SOURCE {baseline_source_run_id} -> GENERATED {run_id}; generation {generation_id}")
    fidelity_metrics = spark.table(f"{PROFILE_SCHEMA}.fidelity_metrics").filter(
        (F.col("source_run_id") == baseline_source_run_id) & (F.col("generated_run_id") == run_id)
    )
    print("Metric availability by family")
    display(fidelity_metrics.groupBy("family", "comparison_status").count().orderBy("family", "comparison_status"))
    print("Largest deviations within each family and ranking basis")
    display(fidelity_metrics.filter(F.col("highlight")).orderBy("family", "ranking_basis", "deviation_rank"))
    print("Unavailable comparisons and generated integrity issues")
    display(fidelity_metrics.filter(
        (F.col("comparison_status") == "NOT_COMPARABLE") | (F.col("generated_check_status") == "ISSUE")
    ).orderBy("family", "entity_name", "column_name", "metric"))
    print("All individual metrics")
    display(fidelity_metrics.orderBy("family", "entity_name", "column_name", "metric", "dimensions_json"))
    fidelity_output_dir = dbutils.widgets.get("fidelity_output_dir").strip()
    if fidelity_output_dir:
        print(f"Fidelity JSON exported: {export_fidelity_report(fidelity_report, fidelity_output_dir)}")

## Refresh the project showcase

After GENERATED profiling completes, export aggregate comparisons into the static showcase. A blank `showcase_output_dir` updates this checkout's `docs/`. Set an absolute writable folder, such as a test Volume directory, to export a separate HTML/CSS copy. Keep `docs/` and `src/` in the uploaded project. Refresh the exported `index.html` to view results. Commit and push the refreshed files to update GitHub Pages.


In [0]:
%python
if profile_type == "GENERATED":
    from synthetic_banking.showcase import export_showcase

    showcase_output_dir = dbutils.widgets.get("showcase_output_dir").strip()
    showcase_template_dir = src_dir.parent / "docs"
    showcase_output_path = export_showcase(
        spark,
        showcase_output_dir or showcase_template_dir,
        template_dir=showcase_template_dir,
    )
    print(f"Showcase refreshed with aggregate measurements: {showcase_output_path}")
